In [1]:
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import joblib

PATH = r"D:\TA\nih-chest-xrays"
df = pd.read_csv(os.path.join(PATH, "Data_Entry_2017.csv"))

# Label biner
df["label"] = df["Finding Labels"].apply(
    lambda x: 0 if x.strip() == "No Finding" else 1
)

# Fitur tabular
features = ["Patient Age", "Patient Gender", "View Position", "Follow-up #"]
df_feat = df[features + ["Image Index", "label"]].copy()

# Imputasi missing value
df_feat["Patient Age"] = df_feat["Patient Age"].fillna(df_feat["Patient Age"].median())
df_feat["Patient Age"] = df_feat["Patient Age"].clip(upper=100)  # outlier usia > 100
df_feat["Patient Gender"] = df_feat["Patient Gender"].fillna("M")
df_feat["View Position"] = df_feat["View Position"].fillna("PA")
df_feat["Follow-up #"] = df_feat["Follow-up #"].fillna(0)

# Encoding kategorikal
le_gender = LabelEncoder()
df_feat["gender_enc"] = le_gender.fit_transform(df_feat["Patient Gender"])

le_view = LabelEncoder()
df_feat["view_enc"] = le_view.fit_transform(df_feat["View Position"])

# Fitur final
features_final = ["Patient Age", "gender_enc", "view_enc", "Follow-up #"]

X_tab = df_feat[features_final].values.astype(np.float32)
y     = df_feat["label"].values.astype(np.float32)
img_files = df_feat["Image Index"].values

# Split 80:20 stratified
X_tab_train, X_tab_test, y_train, y_test, img_train, img_test = train_test_split(
    X_tab, y, img_files,
    test_size=0.2, random_state=42, stratify=y
)

# Normalisasi tabular
scaler = StandardScaler()
X_tab_train_scaled = scaler.fit_transform(X_tab_train)
X_tab_test_scaled  = scaler.transform(X_tab_test)

# Simpan semua
os.makedirs("../data/processed", exist_ok=True)
pd.DataFrame(X_tab_train_scaled, columns=features_final).to_csv("../data/processed/X_tabular_train.csv", index=False)
pd.DataFrame(X_tab_test_scaled,  columns=features_final).to_csv("../data/processed/X_tabular_test.csv",  index=False)
pd.DataFrame(y_train, columns=["label"]).to_csv("../data/processed/y_train.csv", index=False)
pd.DataFrame(y_test,  columns=["label"]).to_csv("../data/processed/y_test.csv",  index=False)
pd.DataFrame(img_train, columns=["Image Index"]).to_csv("../data/processed/img_train.csv", index=False)
pd.DataFrame(img_test,  columns=["Image Index"]).to_csv("../data/processed/img_test.csv",  index=False)
joblib.dump(scaler, "../models/scaler_tabular.pkl")

print("Semua data processed berhasil disimpan!")
print(f"Train: {len(y_train)} | Test: {len(y_test)}")

Semua data processed berhasil disimpan!
Train: 89696 | Test: 22424
